# Descripción del conjunto final

En este cuaderno se describe el conjunto que resulta del preprocesamiento: los tres años de Ausgrid unidos con ERA5 en un solo archivo, `data/processed/ausgrid_era5_joined.parquet`. El objetivo es explicar qué contiene cada fila y cada columna y cómo se usan las marcas de calidad. No se hace un análisis exploratorio.

El archivo se genera ejecutando, desde la raíz del proyecto y en este orden:

```
python -m src.preprocessing.weather
python -m src.preprocessing.ausgrid
python -m src.preprocessing.merge
```

Las decisiones que hay detrás de cada paso están en `notebooks/weather_exploration.ipynb` y `notebooks/ausgrid_exploration.ipynb`.

## 1. Carga y estructura

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "processed" / "ausgrid_era5_joined.parquet"
if not data_path.is_file():
    raise FileNotFoundError(f"No se encontró {data_path}. Ejecutar antes los scripts de src/preprocessing.")

data = pd.read_parquet(data_path)
print("Filas:", f"{len(data):,}")
print("Columnas:", data.shape[1])
print("Memoria en pandas:", round(data.memory_usage(deep=True).sum() / 1e9, 2), "GB")
print("Clave cliente + hora UTC única:", not data.duplicated(["customer", "timestamp_utc"]).any())
data.head()

Filas: 7,886,556
Columnas: 21
Memoria en pandas: 1.01 GB


Clave cliente + hora UTC única: True


,customer,timestamp_utc,source_period,generator_capacity_kwp,postcode,gc_kwh,cl_kwh,gg_kwh,demand_kwh,has_cl,...,gg_unreliable,temperature_2m,relative_humidity_2m,precipitation,cloud_cover,shortwave_radiation,direct_normal_irradiance,diffuse_radiation,wind_speed_10m_ms,surface_pressure
0,1,2010-07-01 00:00:00+00:00,2010-2011,3.78,2076,0.566,0.0,0.656,0.566,True,...,False,7.5,77,0.0,35,320.0,526.8,116.0,1.416667,1014.6
1,1,2010-07-01 01:00:00+00:00,2010-2011,3.78,2076,1.459,0.0,1.351,1.459,True,...,False,11.9,59,0.0,10,446.0,698.7,106.0,1.527778,1014.1
2,1,2010-07-01 02:00:00+00:00,2010-2011,3.78,2076,2.948,0.0,1.407,2.948,True,...,False,14.4,49,0.0,8,518.0,782.5,97.0,1.611111,1013.6
3,1,2010-07-01 03:00:00+00:00,2010-2011,3.78,2076,3.361,0.0,2.175,3.361,True,...,False,14.7,48,0.0,2,527.0,834.8,78.0,1.388889,1012.2
4,1,2010-07-01 04:00:00+00:00,2010-2011,3.78,2076,1.649,0.0,2.331,1.649,True,...,False,15.3,46,0.0,9,461.0,798.0,73.0,1.222222,1012.2


**Resultado.** Cada fila corresponde a **un cliente en una hora**. La hora está en `timestamp_utc`, en UTC, y marca el **final** de la hora: la fila de las 05:00 UTC contiene la energía consumida y generada entre las 04:00 y las 05:00. No hay dos filas con el mismo cliente y la misma hora.

## 2. Columnas

El conjunto tiene 21 columnas en cuatro grupos: identificación, energía de Ausgrid, marcas de calidad y variables meteorológicas de ERA5.

In [2]:
column_info = pd.DataFrame(
    [
        ("customer", "Identificación", "Identificador del hogar (1 a 300)", "—"),
        ("timestamp_utc", "Identificación", "Final de la hora, en UTC", "—"),
        ("source_period", "Identificación", "Archivo de Ausgrid de origen (2010-2011, 2011-2012 o 2012-2013)", "—"),
        ("generator_capacity_kwp", "Identificación", "Potencia fotovoltaica instalada", "kWp"),
        ("postcode", "Identificación", "Código postal del hogar", "—"),
        ("gc_kwh", "Energía", "Consumo general (GC) de la hora", "kWh"),
        ("cl_kwh", "Energía", "Consumo de la carga controlada (CL) de la hora; vacío si el hogar no tiene esa carga", "kWh"),
        ("gg_kwh", "Energía", "Generación fotovoltaica bruta (GG) de la hora", "kWh"),
        ("demand_kwh", "Energía", "Consumo total del hogar: GC + CL, o GC si no tiene carga controlada", "kWh"),
        ("has_cl", "Marca", "El hogar usa carga controlada en ese año", "—"),
        ("cl_estimated", "Marca", "El CL de la hora se copió del mismo día del año anterior", "—"),
        ("gg_unreliable", "Marca", "La generación de esa hora no es fiable (capacidad mal registrada o reloj desplazado)", "—"),
        ("temperature_2m", "ERA5", "Temperatura del aire a 2 m, instantánea al final de la hora", "°C"),
        ("relative_humidity_2m", "ERA5", "Humedad relativa a 2 m, instantánea", "%"),
        ("precipitation", "ERA5", "Precipitación acumulada en la hora", "mm"),
        ("cloud_cover", "ERA5", "Cobertura nubosa total, instantánea", "%"),
        ("shortwave_radiation", "ERA5", "Irradiancia global horizontal (GHI), promedio de la hora", "W/m²"),
        ("direct_normal_irradiance", "ERA5", "Irradiancia directa normal (DNI), promedio de la hora", "W/m²"),
        ("diffuse_radiation", "ERA5", "Irradiancia difusa horizontal (DHI), promedio de la hora", "W/m²"),
        ("wind_speed_10m_ms", "ERA5", "Velocidad del viento a 10 m, instantánea", "m/s"),
        ("surface_pressure", "ERA5", "Presión en superficie, instantánea", "hPa"),
    ],
    columns=["columna", "grupo", "descripción", "unidad"],
).set_index("columna")
column_info["tipo de dato"] = data.dtypes.astype(str)
column_info["celdas vacías"] = data.isna().sum()
column_info

,grupo,descripción,unidad,tipo de dato,celdas vacías
columna,,,,,
customer,Identificación,Identificador del hogar (1 a 300),—,int16,0
timestamp_utc,Identificación,"Final de la hora, en UTC",—,"datetime64[ns, UTC]",0
source_period,Identificación,"Archivo de Ausgrid de origen (2010-2011, 2011-...",—,category,0
generator_capacity_kwp,Identificación,Potencia fotovoltaica instalada,kWp,float64,0
postcode,Identificación,Código postal del hogar,—,int16,0
gc_kwh,Energía,Consumo general (GC) de la hora,kWh,float64,0
cl_kwh,Energía,Consumo de la carga controlada (CL) de la hora...,kWh,float64,4246336
gg_kwh,Energía,Generación fotovoltaica bruta (GG) de la hora,kWh,float64,0
demand_kwh,Energía,"Consumo total del hogar: GC + CL, o GC si no t...",kWh,float64,0


**Resultado.** La única columna con celdas vacías es `cl_kwh`. Está vacía en los hogares que no tienen carga controlada: no es un dato perdido, sino una carga que no existe. Para el consumo se usa siempre `demand_kwh`, que no tiene vacíos.

## 3. Clientes y período

Se cuenta cuántos clientes hay, cuántas horas tiene cada uno y qué período cubre el conjunto, en UTC y en hora local de Sídney.

In [3]:
hours_per_customer = data.groupby("customer").size()
print("Clientes:", data["customer"].nunique())
print("Horas por cliente:", hours_per_customer.value_counts().to_dict())
print("Cliente con menos horas:", int(hours_per_customer.idxmin()))
print()
first, last = data["timestamp_utc"].min(), data["timestamp_utc"].max()
print("Primera hora (UTC):", first, "| hora local:", first.tz_convert("Australia/Sydney"))
print("Última hora (UTC): ", last, "| hora local:", last.tz_convert("Australia/Sydney"))
print()
print("Filas por archivo de origen:")
data["source_period"].value_counts(sort=False)

Clientes: 300
Horas por cliente: {26295: 299, 24351: 1}
Cliente con menos horas: 2



Primera hora (UTC): 2010-07-01 00:00:00+00:00 | hora local: 2010-07-01 10:00:00+10:00
Última hora (UTC):  2013-06-30 14:00:00+00:00 | hora local: 2013-07-01 00:00:00+10:00

Filas por archivo de origen:


source_period
2010-2011    2625300
2011-2012    2635200
2012-2013    2626056
Name: count, dtype: int64

**Resultado.** Hay 300 clientes. Casi todos tienen 26.295 horas seguidas: la primera es la que termina a las 10:00 hora local del 1 de julio de 2010 y la última la que termina a la medianoche del 30 de junio de 2013. El cliente 2 tiene menos porque le faltan los 81 días del 12 de octubre al 31 de diciembre de 2012, que no existen en los datos originales y no se rellenaron.

Las 9 primeras horas del 1 de julio de 2010 se eliminaron al unir, porque ERA5 no las cubre.

## 4. Marcas de calidad

Las marcas no cambian ningún valor. Indican qué filas conviene tratar aparte según el uso.

In [4]:
flags = pd.DataFrame(
    [
        ("has_cl", "El hogar usa carga controlada en ese año",
         "Solo informativa: demand_kwh ya la tiene en cuenta"),
        ("cl_estimated", "CL copiado del mismo día del año anterior (10 hogares, febrero y marzo de 2013)",
         "Se puede usar para entrenar; no usar para evaluar los modelos de demanda"),
        ("gg_unreliable", "15 hogares con GG por encima de su capacidad y 131 días con el reloj desplazado (hogares 97 y 146)",
         "Excluir al validar la generación calculada con pvlib; no afecta a la demanda"),
    ],
    columns=["marca", "significado", "cómo usarla"],
).set_index("marca")
flags["filas marcadas"] = [int(data[flag].sum()) for flag in flags.index]
flags["porcentaje"] = (100 * flags["filas marcadas"] / len(data)).round(2)
flags

,significado,cómo usarla,filas marcadas,porcentaje
marca,,,,
has_cl,El hogar usa carga controlada en ese año,Solo informativa: demand_kwh ya la tiene en cu...,3565416,45.21
cl_estimated,CL copiado del mismo día del año anterior (10 ...,Se puede usar para entrenar; no usar para eval...,13680,0.17
gg_unreliable,15 hogares con GG por encima de su capacidad y...,Excluir al validar la generación calculada con...,397569,5.04


**Resultado.** Todas las horas están completas y todos los hogares con carga controlada tienen su `CL`, registrado o estimado; si no fuera así, `ausgrid.py` se detendría con un error, por lo que no hacen falta marcas para esos casos. `has_cl` vale `True` en algo menos de la mitad de las filas, porque solo 135 o 136 hogares por año usan carga controlada.

## 5. Rangos de valores

Resumen de las columnas de energía y de clima para tener una referencia de sus órdenes de magnitud.

In [5]:
numeric_columns = ["generator_capacity_kwp", "gc_kwh", "cl_kwh", "gg_kwh", "demand_kwh",
                   "temperature_2m", "relative_humidity_2m", "precipitation", "cloud_cover",
                   "shortwave_radiation", "direct_normal_irradiance", "diffuse_radiation",
                   "wind_speed_10m_ms", "surface_pressure"]
summary = data[numeric_columns].describe().T[["min", "50%", "mean", "max"]]
summary.columns = ["mínimo", "mediana", "media", "máximo"]
summary.insert(0, "unidad", column_info.loc[numeric_columns, "unidad"])
summary.round(3)

,unidad,mínimo,mediana,media,máximo
generator_capacity_kwp,kWp,1.0,1.500,1.683,9.990
gc_kwh,kWh,0.0,0.419,0.662,14.117
cl_kwh,kWh,0.0,0.000,0.210,10.092
gg_kwh,kWh,0.0,0.000,0.243,8.663
demand_kwh,kWh,0.0,0.453,0.759,14.117
temperature_2m,°C,1.3,17.000,16.954,44.800
relative_humidity_2m,%,10.0,76.000,72.545,100.000
precipitation,mm,0.0,0.000,0.086,9.000
cloud_cover,%,0.0,49.000,50.316,100.000
shortwave_radiation,W/m²,0.0,7.000,187.335,1111.000


**Resultado.** Un hogar consume de mediana unos 0,5 kWh por hora. Los máximos de consumo y generación son los mismos picos revisados en el cuaderno de Ausgrid, ahora sumados por hora. Las variables de clima tienen los mismos rangos que en el cuaderno de ERA5, con el viento ya en m/s.

## 6. Ejemplo de un cliente

Las primeras horas del cliente 1, con la hora local al lado para leerlas con más facilidad.

In [6]:
example = data.loc[data["customer"].eq(1)].head(6).copy()
example.insert(2, "hora_local", example["timestamp_utc"].dt.tz_convert("Australia/Sydney"))
example

,customer,timestamp_utc,hora_local,source_period,generator_capacity_kwp,postcode,gc_kwh,cl_kwh,gg_kwh,demand_kwh,...,gg_unreliable,temperature_2m,relative_humidity_2m,precipitation,cloud_cover,shortwave_radiation,direct_normal_irradiance,diffuse_radiation,wind_speed_10m_ms,surface_pressure
0,1,2010-07-01 00:00:00+00:00,2010-07-01 10:00:00+10:00,2010-2011,3.78,2076,0.566,0.0,0.656,0.566,...,False,7.5,77,0.0,35,320.0,526.8,116.0,1.416667,1014.6
1,1,2010-07-01 01:00:00+00:00,2010-07-01 11:00:00+10:00,2010-2011,3.78,2076,1.459,0.0,1.351,1.459,...,False,11.9,59,0.0,10,446.0,698.7,106.0,1.527778,1014.1
2,1,2010-07-01 02:00:00+00:00,2010-07-01 12:00:00+10:00,2010-2011,3.78,2076,2.948,0.0,1.407,2.948,...,False,14.4,49,0.0,8,518.0,782.5,97.0,1.611111,1013.6
3,1,2010-07-01 03:00:00+00:00,2010-07-01 13:00:00+10:00,2010-2011,3.78,2076,3.361,0.0,2.175,3.361,...,False,14.7,48,0.0,2,527.0,834.8,78.0,1.388889,1012.2
4,1,2010-07-01 04:00:00+00:00,2010-07-01 14:00:00+10:00,2010-2011,3.78,2076,1.649,0.0,2.331,1.649,...,False,15.3,46,0.0,9,461.0,798.0,73.0,1.222222,1012.2
5,1,2010-07-01 05:00:00+00:00,2010-07-01 15:00:00+10:00,2010-2011,3.78,2076,2.917,0.0,1.694,2.917,...,False,15.4,45,0.0,40,340.0,701.0,69.0,1.055556,1012.6


## 7. Notas para usar el conjunto

- **Hora.** `timestamp_utc` es el final de la hora en UTC. Para trabajar en hora local se usa `data["timestamp_utc"].dt.tz_convert("Australia/Sydney")`, que ya aplica el horario de verano.
- **Variable objetivo de demanda.** `demand_kwh`, que incluye la carga controlada. `gc_kwh` y `cl_kwh` se conservan por separado.
- **Generación.** `gg_kwh` es la generación medida. Al compararla con pvlib se excluyen las filas con `gg_unreliable`.
- **Radiación.** Las tres variables de radiación son el promedio de la hora que termina en `timestamp_utc`. En pvlib la posición del sol se calcula a mitad de esa hora, es decir, en `timestamp_utc - 30 min`.
- **Evaluación.** Las filas con `cl_estimated` no se usan para medir el error de los modelos de demanda.
- **Clima.** ERA5 corresponde a un solo punto, el centro de Sídney, y se repite para todos los hogares de la misma hora, aunque estén en códigos postales distintos.